In [13]:
import os
import re
import json
import time
import requests
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset

In [14]:
# Output folders
BASE_DIR = "/home/shayan/Distributional-Hate-Speech-Prediction/data"
ARTICLE_DIR = os.path.join(BASE_DIR, "wikimedia_articles")
SUMMARY_DIR = os.path.join(BASE_DIR, "summary_bank")

os.makedirs(ARTICLE_DIR, exist_ok=True)
os.makedirs(SUMMARY_DIR, exist_ok=True)

RAW_ARTICLES_PATH = os.path.join(ARTICLE_DIR, "pilot_selected_articles_raw.jsonl")
CLEAN_ARTICLES_PATH = os.path.join(ARTICLE_DIR, "pilot_selected_articles_clean.jsonl")
ALL_SUMMARIES_PATH = os.path.join(SUMMARY_DIR, "pilot_summaries_all_models.jsonl")
QUAL_ANALYSIS_PATH = os.path.join(SUMMARY_DIR, "pilot_qualitative_analysis.csv")

print("Folders ready.")

Folders ready.


In [15]:
PILOT_ARTICLES = [
    {
        "summary_id": "WIKI_WOMEN_MISOGYNY_001",
        "identity_group": "women",
        "topic": "misogyny",
        "preferred_title": "Misogyny",
        "title_aliases": ["Misogyny", "Misogynist", "Misogyny and mass media"]
    },
    {
        "summary_id": "WIKI_WOMEN_SEXISM_001",
        "identity_group": "women",
        "topic": "sexism",
        "preferred_title": "Sexism",
        "title_aliases": ["Sexism", "Gender discrimination", "Gender inequality"]
    },
    {
        "summary_id": "WIKI_IMM_XENOPHOBIA_001",
        "identity_group": "immigrants",
        "topic": "xenophobia",
        "preferred_title": "Xenophobia",
        "title_aliases": ["Xenophobia", "Racism", "Ethnic hatred"]
    },
    {
        "summary_id": "WIKI_IMM_OPPOSITION_IMMIGRATION_001",
        "identity_group": "immigrants",
        "topic": "opposition to immigration",
        "preferred_title": "Opposition to immigration",
        "title_aliases": ["Opposition to immigration", "Anti-immigration", "Nativism (politics)"]
    }
]

print("Pilot article targets:")
for item in PILOT_ARTICLES:
    print(item["topic"], "=>", item["title_aliases"])

Pilot article targets:
misogyny => ['Misogyny', 'Misogynist', 'Misogyny and mass media']
sexism => ['Sexism', 'Gender discrimination', 'Gender inequality']
xenophobia => ['Xenophobia', 'Racism', 'Ethnic hatred']
opposition to immigration => ['Opposition to immigration', 'Anti-immigration', 'Nativism (politics)']


In [16]:
wiki_stream = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True
)

print("Wikimedia stream loaded.")

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Wikimedia stream loaded.


In [17]:
alias_to_meta = {}

for item in PILOT_ARTICLES:
    for alias in item["title_aliases"]:
        alias_to_meta[alias.lower()] = item

wanted_aliases_lower = set(alias_to_meta.keys())

wanted_aliases_lower

{'anti-immigration',
 'ethnic hatred',
 'gender discrimination',
 'gender inequality',
 'misogynist',
 'misogyny',
 'misogyny and mass media',
 'nativism (politics)',
 'opposition to immigration',
 'racism',
 'sexism',
 'xenophobia'}

In [18]:
# IMPORTANT: Reload stream before searching again
wiki_stream = load_dataset(
    "wikimedia/wikipedia",
    "20231101.en",
    split="train",
    streaming=True
)

selected_articles = {}

for row in tqdm(wiki_stream, desc="Searching Wikimedia articles"):
    title = row.get("title", "")
    title_lower = title.lower()

    # exact alias match
    if title_lower in wanted_aliases_lower:
        meta = alias_to_meta[title_lower]
        summary_id = meta["summary_id"]

        if summary_id not in selected_articles:
            selected_articles[summary_id] = {
                "summary_id": meta["summary_id"],
                "identity_group": meta["identity_group"],
                "topic": meta["topic"],
                "preferred_title": meta["preferred_title"],
                "matched_title": title,
                "source_dataset": "wikimedia/wikipedia",
                "source_config": "20231101.en",
                "source_title": title,
                "source_url": row.get("url", ""),
                "source_article_id": row.get("id", ""),
                "raw_text": row.get("text", "")
            }

            print(f"Found: {title}  -> topic: {meta['topic']}")

    if len(selected_articles) == len(PILOT_ARTICLES):
        break

print(f"\nTotal found: {len(selected_articles)} / {len(PILOT_ARTICLES)}")
print([article["source_title"] for article in selected_articles.values()])

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Searching Wikimedia articles: 0it [00:00, ?it/s]

Found: Opposition to immigration  -> topic: opposition to immigration
Found: Ethnic hatred  -> topic: xenophobia
Found: Gender inequality  -> topic: sexism
Found: Misogyny and mass media  -> topic: misogyny

Total found: 4 / 4
['Opposition to immigration', 'Ethnic hatred', 'Gender inequality', 'Misogyny and mass media']


In [19]:
with open(RAW_ARTICLES_PATH, "w", encoding="utf-8") as f:
    for article in selected_articles.values():
        f.write(json.dumps(article, ensure_ascii=False) + "\n")

print(f"Saved raw articles to: {RAW_ARTICLES_PATH}")

Saved raw articles to: /home/shayan/Distributional-Hate-Speech-Prediction/data/wikimedia_articles/pilot_selected_articles_raw.jsonl


In [20]:
def clean_wiki_text(text):
    """
    Basic Wikipedia cleanup:
    - remove reference-heavy tail sections
    - normalize whitespace
    - keep main body
    """
    if not isinstance(text, str):
        return ""

    stop_sections = [
        "References",
        "See also",
        "External links",
        "Further reading",
        "Bibliography",
        "Notes"
    ]

    for section in stop_sections:
        pattern = rf"\n\s*{re.escape(section)}\s*\n"
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            text = text[:match.start()]
            break

    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()

    return text


clean_articles = []

for article in selected_articles.values():
    clean_text = clean_wiki_text(article["raw_text"])

    clean_article = {
        **{k: v for k, v in article.items() if k != "raw_text"},
        "clean_text": clean_text,
        "char_count": len(clean_text),
        "word_count": len(clean_text.split())
    }

    clean_articles.append(clean_article)

print(f"Cleaned {len(clean_articles)} articles.")

Cleaned 4 articles.


In [21]:
preview_df = pd.DataFrame([
    {
        "summary_id": a["summary_id"],
        "identity_group": a["identity_group"],
        "topic": a["topic"],
        "source_title": a["source_title"],
        "word_count": a["word_count"],
        "source_url": a["source_url"]
    }
    for a in clean_articles
])

preview_df

,summary_id,identity_group,topic,source_title,word_count,source_url
0,WIKI_IMM_OPPOSITION_IMMIGRATION_001,immigrants,opposition to immigration,Opposition to immigration,9940,https://en.wikipedia.org/wiki/Opposition%20to%...
1,WIKI_IMM_XENOPHOBIA_001,immigrants,xenophobia,Ethnic hatred,853,https://en.wikipedia.org/wiki/Ethnic%20hatred
2,WIKI_WOMEN_SEXISM_001,women,sexism,Gender inequality,10362,https://en.wikipedia.org/wiki/Gender%20inequality
3,WIKI_WOMEN_MISOGYNY_001,women,misogyny,Misogyny and mass media,2690,https://en.wikipedia.org/wiki/Misogyny%20and%2...


In [22]:
with open(CLEAN_ARTICLES_PATH, "w", encoding="utf-8") as f:
    for article in clean_articles:
        f.write(json.dumps(article, ensure_ascii=False) + "\n")

print(f"Saved cleaned articles to: {CLEAN_ARTICLES_PATH}")

Saved cleaned articles to: /home/shayan/Distributional-Hate-Speech-Prediction/data/wikimedia_articles/pilot_selected_articles_clean.jsonl


In [23]:
def build_summary_prompt(article, max_source_chars=7000):
    article_text = article["clean_text"][:max_source_chars]

    prompt = f"""
You are generating background knowledge summaries for a hate speech detection research project.

Identity group:
{article["identity_group"]}

Article title:
{article["source_title"]}

Topic:
{article["topic"]}

Source article text:
{article_text}

Task:
Write a concise background knowledge summary grounded only in the provided article text.

The summary should explain relevant social context, stereotypes, hostile narratives, identity-based prejudice, or discourse patterns that may help a classifier interpret online hate speech.

Rules:
- Use only the provided article text.
- Do not classify any specific post.
- Do not say "this is hate speech".
- Do not instruct the classifier what label to predict.
- Do not include moral judgement.
- Do not add unsupported information.
- Keep the summary between 50 and 90 words.
- Write one paragraph only.

Summary:
""".strip()

    return prompt

In [24]:
OLLAMA_URL = "http://127.0.0.1:11434/api/generate"

def ollama_generate(prompt, model_name, temperature=0.2, max_retries=3):
    payload = {
        "model": model_name,
        "prompt": prompt,
        "stream": False,
        "keep_alive": "5m",
        "options": {
            "temperature": temperature,
            "num_predict": 180
        }
    }

    for attempt in range(max_retries):
        try:
            response = requests.post(OLLAMA_URL, json=payload, timeout=240)
            response.raise_for_status()
            data = response.json()
            return data.get("response", "").strip()

        except Exception as e:
            print(f"Attempt {attempt + 1} failed for {model_name}: {e}")
            time.sleep(3)

    return ""

In [25]:
test_prompt = "Summarise this in one sentence: Wikipedia is a free online encyclopedia."

test_response = ollama_generate(test_prompt, model_name="qwen2.5:7b-instruct")

print(test_response)

Wikipedia is an online encyclopedia that provides freely editable and distributed content created and maintained by volunteers from around the world.


In [26]:
SUMMARISATION_MODELS = [
    "qwen2.5:7b-instruct",
    "llama3.1:8b",
    "mistral:7b"
]

SUMMARISATION_MODELS

['qwen2.5:7b-instruct', 'llama3.1:8b', 'mistral:7b']

In [27]:
import subprocess
import gc

def stop_ollama_model(model_name):
    """
    Unload a model from Ollama memory after use.
    Works if your Ollama version supports `ollama stop`.
    """
    try:
        result = subprocess.run(
            ["ollama", "stop", model_name],
            capture_output=True,
            text=True
        )
        print(f"Stopped {model_name}: {result.stdout.strip() or result.stderr.strip()}")
    except Exception as e:
        print(f"Could not stop {model_name}: {e}")


all_summaries = []

# If previous output already exists, load it so we don't duplicate work
if os.path.exists(ALL_SUMMARIES_PATH):
    with open(ALL_SUMMARIES_PATH, "r", encoding="utf-8") as f:
        all_summaries = [json.loads(line) for line in f if line.strip()]

done_keys = {
    (item["summary_id"], item["model_name"])
    for item in all_summaries
}

print(f"Already completed: {len(done_keys)} summaries")


for model_name in SUMMARISATION_MODELS:
    print("=" * 100)
    print(f"Starting model: {model_name}")
    print("=" * 100)

    for article in tqdm(clean_articles, desc=f"Summarising with {model_name}"):

        key = (article["summary_id"], model_name)

        if key in done_keys:
            print(f"Skipping existing: {article['source_title']} with {model_name}")
            continue

        prompt = build_summary_prompt(article)

        summary_text = ollama_generate(
            prompt=prompt,
            model_name=model_name,
            temperature=0.2
        )

        record = {
            "summary_id": article["summary_id"],
            "identity_group": article["identity_group"],
            "topic": article["topic"],
            "source_dataset": article["source_dataset"],
            "source_config": article["source_config"],
            "source_title": article["source_title"],
            "source_url": article["source_url"],
            "source_article_id": article["source_article_id"],
            "model_name": model_name,
            "summary_text": summary_text,
            "summary_word_count": len(summary_text.split()),
            "selected": False,
            "notes": ""
        }

        all_summaries.append(record)
        done_keys.add(key)

        # Save after every single summary
        with open(ALL_SUMMARIES_PATH, "w", encoding="utf-8") as f:
            for item in all_summaries:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

        print(f"Saved: {article['source_title']} | {model_name} | {len(summary_text.split())} words")

    # Unload this model before moving to the next one
    stop_ollama_model(model_name)
    gc.collect()

print("=" * 100)
print(f"Finished. Generated/stored {len(all_summaries)} summaries.")
print(f"Saved to: {ALL_SUMMARIES_PATH}")

Already completed: 1 summaries
Starting model: qwen2.5:7b-instruct


Summarising with qwen2.5:7b-instruct:   0%|          | 0/4 [00:00<?, ?it/s]

Skipping existing: Opposition to immigration with qwen2.5:7b-instruct
Saved: Ethnic hatred | qwen2.5:7b-instruct | 64 words
Saved: Gender inequality | qwen2.5:7b-instruct | 72 words
Saved: Misogyny and mass media | qwen2.5:7b-instruct | 48 words
Stopped qwen2.5:7b-instruct: 
Starting model: llama3.1:8b


Summarising with llama3.1:8b:   0%|          | 0/4 [00:00<?, ?it/s]

Saved: Opposition to immigration | llama3.1:8b | 76 words
Saved: Ethnic hatred | llama3.1:8b | 74 words
Saved: Gender inequality | llama3.1:8b | 73 words
Saved: Misogyny and mass media | llama3.1:8b | 72 words
Stopped llama3.1:8b: 
Starting model: mistral:7b


Summarising with mistral:7b:   0%|          | 0/4 [00:00<?, ?it/s]

Saved: Opposition to immigration | mistral:7b | 79 words
Saved: Ethnic hatred | mistral:7b | 120 words
Saved: Gender inequality | mistral:7b | 114 words
Saved: Misogyny and mass media | mistral:7b | 102 words
Stopped mistral:7b: 
Finished. Generated/stored 12 summaries.
Saved to: /home/shayan/Distributional-Hate-Speech-Prediction/data/summary_bank/pilot_summaries_all_models.jsonl


In [28]:
summary_df = pd.read_json(ALL_SUMMARIES_PATH, lines=True)

summary_df[
    [
        "summary_id",
        "identity_group",
        "topic",
        "source_title",
        "model_name",
        "summary_word_count",
        "summary_text"
    ]
]

,summary_id,identity_group,topic,source_title,model_name,summary_word_count,summary_text
0,WIKI_IMM_OPPOSITION_IMMIGRATION_001,immigrants,opposition to immigration,Opposition to immigration,qwen2.5:7b-instruct,58,The article discusses various arguments agains...
1,WIKI_IMM_XENOPHOBIA_001,immigrants,xenophobia,Ethnic hatred,qwen2.5:7b-instruct,64,Ethnic hatred involves prejudice and hostility...
2,WIKI_WOMEN_SEXISM_001,women,sexism,Gender inequality,qwen2.5:7b-instruct,72,"The article discusses gender inequality, highl..."
3,WIKI_WOMEN_MISOGYNY_001,women,misogyny,Misogyny and mass media,qwen2.5:7b-instruct,48,"The article highlights how misogyny in media, ..."
4,WIKI_IMM_OPPOSITION_IMMIGRATION_001,immigrants,opposition to immigration,Opposition to immigration,llama3.1:8b,76,Opposition to immigration often centers on con...
5,WIKI_IMM_XENOPHOBIA_001,immigrants,xenophobia,Ethnic hatred,llama3.1:8b,74,Ethnic hatred refers to prejudice and hostilit...
6,WIKI_WOMEN_SEXISM_001,women,sexism,Gender inequality,llama3.1:8b,73,Women are disproportionately affected by gende...
7,WIKI_WOMEN_MISOGYNY_001,women,misogyny,Misogyny and mass media,llama3.1:8b,72,"Misogyny in mass media, particularly in music ..."
8,WIKI_IMM_OPPOSITION_IMMIGRATION_001,immigrants,opposition to immigration,Opposition to immigration,mistral:7b,79,The article discusses opposition to immigratio...
9,WIKI_IMM_XENOPHOBIA_001,immigrants,xenophobia,Ethnic hatred,mistral:7b,120,"Ethnic hatred, a form of prejudice and hostili..."


In [29]:
for source_title in summary_df["source_title"].unique():
    print("=" * 100)
    print(f"ARTICLE: {source_title}")
    print("=" * 100)

    temp = summary_df[summary_df["source_title"] == source_title]

    for _, row in temp.iterrows():
        print(f"\nMODEL: {row['model_name']}")
        print("-" * 80)
        print(row["summary_text"])
        print(f"\nWord count: {row['summary_word_count']}")

ARTICLE: Opposition to immigration

MODEL: qwen2.5:7b-instruct
--------------------------------------------------------------------------------
The article discusses various arguments against immigration, including concerns about national identity, social cohesion, economic competition, environmental impacts, health risks, crime rates, military loyalty, dangerous journeys, cultural changes, and welfare costs. These arguments often reflect stereotypes and fears of ethnic or racial groups being excluded, and can lead to hostile narratives that may contribute to prejudice against immigrants.

Word count: 58

MODEL: llama3.1:8b
--------------------------------------------------------------------------------
Opposition to immigration often centers on concerns about national identity, with some arguing that immigrants threaten cultural and racial unity. Hostile narratives also emphasize economic burdens, crime rates, and disease transmission associated with immigration. Additionally, stereot

In [30]:
score_rows = []

for _, row in summary_df.iterrows():
    score_rows.append({
        "summary_id": row["summary_id"],
        "identity_group": row["identity_group"],
        "topic": row["topic"],
        "source_title": row["source_title"],
        "model_name": row["model_name"],
        "groundedness": "",
        "relevance": "",
        "neutrality": "",
        "non_label_leakage": "",
        "clarity": "",
        "conciseness": "",
        "total_score": "",
        "qualitative_notes": ""
    })

score_df = pd.DataFrame(score_rows)
score_df.to_csv(QUAL_ANALYSIS_PATH, index=False)

print(f"Qualitative scoring sheet saved to: {QUAL_ANALYSIS_PATH}")
score_df.head()

Qualitative scoring sheet saved to: /home/shayan/Distributional-Hate-Speech-Prediction/data/summary_bank/pilot_qualitative_analysis.csv


,summary_id,identity_group,topic,source_title,model_name,groundedness,relevance,neutrality,non_label_leakage,clarity,conciseness,total_score,qualitative_notes
0,WIKI_IMM_OPPOSITION_IMMIGRATION_001,immigrants,opposition to immigration,Opposition to immigration,qwen2.5:7b-instruct,,,,,,,,
1,WIKI_IMM_XENOPHOBIA_001,immigrants,xenophobia,Ethnic hatred,qwen2.5:7b-instruct,,,,,,,,
2,WIKI_WOMEN_SEXISM_001,women,sexism,Gender inequality,qwen2.5:7b-instruct,,,,,,,,
3,WIKI_WOMEN_MISOGYNY_001,women,misogyny,Misogyny and mass media,qwen2.5:7b-instruct,,,,,,,,
4,WIKI_IMM_OPPOSITION_IMMIGRATION_001,immigrants,opposition to immigration,Opposition to immigration,llama3.1:8b,,,,,,,,


In [31]:
label_leakage_phrases = [
    "this is hate speech",
    "should be classified",
    "classify this",
    "the post is hateful",
    "this comment is hateful",
    "label as",
    "predict"
]

def check_summary_issues(text):
    text_lower = text.lower()
    issues = []

    if len(text.split()) < 40:
        issues.append("too_short")

    if len(text.split()) > 100:
        issues.append("too_long")

    for phrase in label_leakage_phrases:
        if phrase in text_lower:
            issues.append("possible_label_leakage")
            break

    if not text.strip():
        issues.append("empty_summary")

    return ", ".join(issues)

summary_df["auto_issues"] = summary_df["summary_text"].apply(check_summary_issues)

summary_df[
    [
        "source_title",
        "model_name",
        "summary_word_count",
        "auto_issues",
        "summary_text"
    ]
]

,source_title,model_name,summary_word_count,auto_issues,summary_text
0,Opposition to immigration,qwen2.5:7b-instruct,58,,The article discusses various arguments agains...
1,Ethnic hatred,qwen2.5:7b-instruct,64,,Ethnic hatred involves prejudice and hostility...
2,Gender inequality,qwen2.5:7b-instruct,72,,"The article discusses gender inequality, highl..."
3,Misogyny and mass media,qwen2.5:7b-instruct,48,,"The article highlights how misogyny in media, ..."
4,Opposition to immigration,llama3.1:8b,76,,Opposition to immigration often centers on con...
5,Ethnic hatred,llama3.1:8b,74,,Ethnic hatred refers to prejudice and hostilit...
6,Gender inequality,llama3.1:8b,73,,Women are disproportionately affected by gende...
7,Misogyny and mass media,llama3.1:8b,72,,"Misogyny in mass media, particularly in music ..."
8,Opposition to immigration,mistral:7b,79,,The article discusses opposition to immigratio...
9,Ethnic hatred,mistral:7b,120,too_long,"Ethnic hatred, a form of prejudice and hostili..."


In [32]:
checked_path = os.path.join(SUMMARY_DIR, "pilot_summaries_all_models_checked.csv")

summary_df.to_csv(checked_path, index=False)

print(f"Saved checked summaries to: {checked_path}")

Saved checked summaries to: /home/shayan/Distributional-Hate-Speech-Prediction/data/summary_bank/pilot_summaries_all_models_checked.csv


In [ ]:
SELECTED_MODEL = "qwen2.5:7b"  #  after qualitative analysis

selected_summary_df = summary_df[summary_df["model_name"] == SELECTED_MODEL].copy()
selected_summary_df["selected"] = True
selected_summary_df["selected_model"] = SELECTED_MODEL

selected_path = os.path.join(SUMMARY_DIR, "pilot_summaries_selected.jsonl")

with open(selected_path, "w", encoding="utf-8") as f:
    for item in selected_summary_df.to_dict(orient="records"):
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved selected summaries to: {selected_path}")
selected_summary_df[
    [
        "summary_id",
        "identity_group",
        "topic",
        "source_title",
        "selected_model",
        "summary_word_count",
        "summary_text"
    ]
]